# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [67]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter:  ········


API key set.


In [68]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [69]:
# @title User input needed.

DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section.",
        "Documents can be related to any topic - sports, manga, technology, or nature.",
        "Have a minimum of 4 sentences with each document.",
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),
    "fill_in1": (
        "Artificial Intelligence has rapidly evolved over the past decade.",
        "Large language models can now reason, code, and generate human-like text.",
        "Agents use reasoning frameworks like ReAct to combine thoughts and actions.",
        "Tools allow these models to search the web and retrieve up-to-date facts."
    ),
    "fill_in2": (
        "Space exploration has entered an ambitious new era with private and public missions.",
        "NASA's Artemis program aims to land the next humans on the Moon.",
        "Rovers on Mars continue searching for signs of past microbial life.",
        "New space telescopes provide clear views into the earliest eras of the universe."
    ),
    "fill_in3": (
        "Solar energy is becoming one of the cheapest sources of electricity worldwide.",
        "Photovoltaic panels convert sunlight directly into clean electrical power.",
        "Battery storage technology allows surplus power to be stored for nighttime use.",
        "Modern power grids are modernizing to support fluctuating renewable inputs."
    ),
    "fill_in4": (
        "Basketball is a fast-paced team sport played on a rectangular court.",
        "Two teams of five players each compete to score points through an elevated hoop.",
        "Points are awarded based on whether a shot is taken inside or outside the arc.",
        "Defensive strategies often alternate between zone and man-to-man coverage."
    )
}

In [70]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        # Ensure content is a string if stored as a tuple/list
        text = " ".join(content) if isinstance(content, (tuple, list)) else str(content)
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in text.lower())
        if match_count > 0:
            results.append((match_count, doc_name, text))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, text in results[:2]:
        output.append(f"\n{text}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))



Students must fill in documents in this section. Documents can be related to any topic - sports, manga, technology, or nature. Have a minimum of 4 sentences with each document. Duplicate this for a minimum of 4 documents to test your agent.


Basketball is a fast-paced team sport played on a rectangular court. Two teams of five players each compete to score points through an elevated hoop. Points are awarded based on whether a shot is taken inside or outside the arc. Defensive strategies often alternate between zone and man-to-man coverage.


In [71]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time
import os

# Configure the API key from Colab secrets
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
genai.configure(api_key=api_key)

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    tools=[search_documents],
    system_instruction=(
        """
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        """
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-1.5-flash


In [72]:
import time
from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"))

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find the most relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    # If 3.8 is busy (503), try 3.6-flash, then 3.5-flash-lite
    candidate_models = ["gemini-3.5-flash-lite"]
    
    for selected_model in candidate_models:
        for attempt in range(max_retries):
            try:
                return client.models.generate_content(
                    model=selected_model,
                    contents=contents,
                    config=config
                )
            except Exception as e:
                err = str(e)
                if ("503" in err or "UNAVAILABLE" in err or "429" in err) and attempt < max_retries - 1:
                    wait = 6 * (attempt + 1)
                    print(f"  {selected_model} busy/rate-limited, retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    if selected_model != candidate_models[-1]:
                        print(f"  {selected_model} unavailable, trying {candidate_models[candidate_models.index(selected_model)+1]}...")
                    else:
                        raise
                    break


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn unmodified
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'students share documents topics'})
  Result preview: 
Students must fill in documents in this section. Documents can be related to any topic - sports, ma...

Final answer:
Students are supposed to share documents related to **any topic**, such as:
* Sports
* Manga
* Technology
* Nature

Each document must contain a minimum of 4 sentences, and you need to provide a minimum of 4 documents in total.


In [73]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "What is the goal of NASA's Artemis program?",
    "How many ghosts eat carrots everyday?",
    "What is the most popular food item in Canada?",
    "What is the main purpose of the James Webb Space Telescope?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


Question: What is the goal of NASA's Artemis program?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Artemis program goal NASA'})
  Result preview: 
Space exploration has entered an ambitious new era with private and public missions. NASA's Artemis...

Final answer:
The goal of NASA's Artemis program is to land the next humans on the Moon.

Question: How many ghosts eat carrots everyday?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'ghosts eat carrots everyday'})
  Result preview: No relevant documents found for that query....
Step 2: model calls search_documents({'query': 'ghosts carrots'})
  Result preview: No relevant documents found for that query....
Step 3: model calls search_documents({'query': 'ghosts'})
  Result preview: No relevant documents found for that query....

Final answer:
Based on the available documents, there is no mention of ghosts eating carrots.

Question: W

In [74]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)